In [4]:
from openai import OpenAI
import gradio as gr
import os
from dotenv import load_dotenv
load_dotenv(override=True)
from scrapper import fetch_website_contents
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini=OpenAI(base_url=gemini_url, api_key=gemini_api_key)

In [5]:
response=gemini.chat.completions.create(
    model='gemini-3.1-flash-lite',
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a short poem about the beauty of nature."}
    ]
)
response_text = response.choices[0].message.content
response_text

'The golden sun wakes up the day,\nWhere velvet meadows softly sway.\nThe rivers hum a crystal song,\nAs ancient forests stretch out long.\n\nThe mountain peaks touch painted skies,\nWhere silent, soaring eagle flies.\nIn every leaf and budding flower,\nResides a calm and healing power.'

In [3]:

# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [6]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),
# Please take a look at the Intermediate Python guide in the guides folder

def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model='gemini-3.1-flash-lite',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [7]:
def stream_broucher(company_name,url):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    result=stream_gemini(prompt)
    yield from result

In [8]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")
view = gr.Interface(
    fn=stream_broucher,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co"],
            ["Edward Donner", "https://edwarddonner.com"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
